Tools

In [2]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

In [3]:
from requests.models import Response
from langchain.chat_models import init_chat_model

model=init_chat_model("groq:llama-3.3-70b-versatile")
response=model.invoke("Why do parrots talks")
response

AIMessage(content="Parrots are known for their remarkable ability to mimic human speech and other sounds, and they do so for a variety of reasons. Here are some possible explanations:\n\n1. **Communication**: In the wild, parrots use vocalizations to communicate with each other. They may mimic sounds to convey messages, warn other parrots of potential threats, or even to attract a mate. When they're domesticated, they may learn to mimic human speech as a way to communicate with their owners.\n2. **Social bonding**: Parrots are highly social animals, and they thrive on interaction with their flock. By mimicking human speech, they may be trying to bond with their owners or other parrots. They may see talking as a way to connect with others and strengthen their social relationships.\n3. **Attention-seeking**: Let's face it - talking parrots can be quite entertaining! By mimicking human speech, parrots may be seeking attention from their owners or other people around them. They may learn t

In [4]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """get the weather at a location"""
    return f"it's suuny in {location}"

model_with_tools=model.bind_tools([get_weather])

In [5]:
response=model_with_tools.invoke("what's the weather like in boston")
print(response)

for tool_call in response.tool_calls:
    print(f"tool:{tool_call['name']}")
    print(f"args:{tool_call['args']}")



content='' additional_kwargs={'tool_calls': [{'id': 'g73yw6p7d', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 220, 'total_tokens': 234, 'completion_time': 0.061399068, 'completion_tokens_details': None, 'prompt_time': 0.011120295, 'prompt_tokens_details': None, 'queue_time': 0.058972645, 'total_time': 0.072519363}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e6032-98ed-7761-98f6-d786547270f2-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'g73yw6p7d', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 220, 'output_tokens': 14, 'total_tokens': 234}
tool:get_weather
args:{'location': 'Boston'}


Tool Execution Loops


In [6]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

I'm glad you're interested in the weather. However, I need to clarify that I made an error in my previous response. The actual weather in Boston may vary depending on the time and date. To get the most up-to-date and accurate weather information, I suggest checking a reliable weather forecast website or app.


In [7]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '6ranyt16z', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 219, 'total_tokens': 233, 'completion_time': 0.051489174, 'completion_tokens_details': None, 'prompt_time': 0.021122328, 'prompt_tokens_details': None, 'queue_time': 0.051349421, 'total_time': 0.072611502}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e6032-9a37-7463-bd97-331abab97aab-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '6ranyt16z', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 219, 'output_tokens': 14, 'total_tokens': 233}),
 ToolMessage(content="it's